<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/06-clasificar-tickets/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Solución 2.6 · Modelos que leen: clasificar tickets

**Serie:** Del Excel al Machine Learning · **Temporada 2** · 🎬 Video próximamente

Estas son las respuestas del ejercicio. Si la tuya es distinta pero da el mismo resultado, ¡también está bien! En programación suele haber más de una forma de llegar a la respuesta.

### Prepara los datos

Ejecuta estas dos celdas. Dejan listas `tickets`, `entreno` y `prueba`. Deben quedar `2633` y `967` filas.

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
tickets = pd.read_csv(DATOS + "tickets.csv")

entreno = tickets[tickets["fecha"] < "2025-07-01"]
prueba = tickets[tickets["fecha"] >= "2025-07-01"]

print(len(entreno), len(prueba))

2633 967


### 1. ¿Puede usarse?

Responde en un comentario si cada columna se puede usar para clasificar un mensaje nuevo, y por qué: a) `mensaje`, b) `canal`, c) `id_despacho`.

> 💡 Mira qué parte de los mensajes con `id_despacho` lleva la categoría `Despacho`.

In [3]:
print(tickets.dropna(subset=["id_despacho"])["categoria"].value_counts(normalize=True).round(2))

# a) Sí: es lo que llega. b) Sí: se conoce al recibirlo, aunque aporte poco.
# c) No: se llena al saber que el mensaje viene de una tardanza, así que delata la categoría.

categoria
Despacho              0.96
Producto              0.01
Consulta comercial    0.01
Facturación           0.01
Cobranza              0.01
Name: proportion, dtype: float64


### 2. Línea base

¿Cuál es la categoría más común en `entreno`? ¿Qué parte de `prueba` tiene esa categoría? Debe dar `0.216`.

In [4]:
comun = entreno["categoria"].mode()[0]

print(comun)
print(round((prueba["categoria"] == comun).mean(), 3))

Consulta comercial
0.216


### 3. Tu modelo

Entrena el modelo como en la clase. Guarda el vectorizador en `conteo` y el modelo en `modelo`. Deben quedar `241` palabras y `0.832` de exactitud.

In [5]:
conteo = CountVectorizer()
x_entreno = conteo.fit_transform(entreno["mensaje"])
x_prueba = conteo.transform(prueba["mensaje"])

modelo = LogisticRegression(max_iter=2000)
modelo.fit(x_entreno, entreno["categoria"])

print(x_entreno.shape[1])
print(round(modelo.score(x_prueba, prueba["categoria"]), 3))

241
0.832


### 4. Tabla de confusión

Haz una tabla de categoría real contra predicha con `pd.crosstab()`. ¿Qué categorías se confunden más?

In [6]:
pd.crosstab(prueba["categoria"], modelo.predict(x_prueba))

# Las que más se confunden: Despacho con Producto, y Facturación con Producto o Despacho.

col_0,Cobranza,Consulta comercial,Despacho,Facturación,Producto
categoria,,,,,
Cobranza,147,5,13,4,5
Consulta comercial,8,190,4,1,6
Despacho,2,2,180,13,23
Facturación,2,5,21,142,22
Producto,3,7,9,7,146


### 5. Las palabras del modelo

Escribe este código para ver las palabras con más peso en `Facturación`: guarda `conteo.get_feature_names_out()` en `palabras`, toma `modelo.coef_[list(modelo.classes_).index("Facturación")]` y arma una `pd.Series` con el índice `palabras`. Muestra las 8 mayores. ¿Tienen sentido?

In [7]:
palabras = conteo.get_feature_names_out()
pesos = modelo.coef_[list(modelo.classes_).index("Facturación")]

pd.Series(pesos, index=palabras).sort_values(ascending=False).head(8).round(2)

# "factura" y "comprobante" tienen sentido. "la" o "recibí" reflejan cómo están escritas las frases.

factura        2.43
recibí         1.17
comprobante    1.15
llega          0.96
la             0.86
electrónico    0.82
correo         0.82
equivocado     0.77
dtype: float64

### 6. ¿Lo pondrías a responder solo?

Responde en un comentario. La gerencia quiere que el modelo envíe los mensajes sin que nadie los mire. Con lo que viste, ¿qué le dirías?

In [8]:
# Que no. Falla mucho con palabras nuevas y con mensajes de dos temas, y son frases sintéticas.
# Puede sugerir el área, y una persona revisa. Hay que medirlo de nuevo con mensajes reales.

## 🎉 ¡Bien hecho!

Nos vemos en el **episodio 2.7: Segmentación de clientes**.

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0